# SAT-13A · Export the 82 test images for relabelling  (CPU, ~2 min)

**Attach:** `sat-ext` + `sat-splits` + `satelite-coco`. **Accelerator:** None.

Run all, then click the link and download **`test_relabel_upload.zip`** (82 test images + their current boxes, COCO).

**Then in Roboflow (your manual step):** *New Project* (Object Detection, classes plane / ship / vehicle) -> *Upload* the
unzipped folder (images **and** `_annotations.coco.json`, so the existing boxes come with them) -> open each of the 82
images and **add every missing plane / ship / vehicle** (most are small cars in car parks and on roads; Label Assist may
propose boxes - check each) -> *Generate Version* with **no preprocessing, no augmentation**, all images in one split ->
*Export* COCO -> upload the unzipped export to Kaggle as Dataset **`sat-test-relabeled`** (name must contain "relabel").
Then run **13B**.

**Input setup** - right panel -> *Add Input* -> search the name -> *Add*. Links (your own Datasets are private: open them from the account that owns them, or share them with this account first):

* `sat-ext` - code (upload the sat_ext folder of the zip): https://www.kaggle.com/datasets/rifat4018/sat-ext
* `satelite-coco` - Roboflow COCO export, 640x640 (or download: universe.roboflow.com/rifat4018/aerial-object-detection-u3f5u): https://www.kaggle.com/datasets/rifat4018/satelite-coco
* `sat-splits` - output of notebook 01: https://www.kaggle.com/datasets/rifat4018/sat-splits
* Check the printed input tree in the next cell before running the rest.

In [ ]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

In [ ]:
utils.warn_if_gpu_wasted()
import relabel
z = relabel.export_for_relabel()
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(z, cfg.WORK_DIR)))
except ImportError:
    pass